# Preliminaries

In this assignment, you will practice working with the kinematics of the 7DOF Kuka iiwa robot. You will perform position control using numerical IK, and you will also experiment with P and PI controllers for velocity inputs.

In [25]:
# Run once at the beginning to install pybullet
!pip install pybullet

In [26]:
import pybullet as p
import pybullet_data
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation

In [81]:
# some utility functions

def plot_joint_progression(solution):
    """Plots the joint angle progression of a given set of joint configurations."""
    joint_configs_array = np.array(solution)
    plt.figure(figsize=(8, 6))
    for i in range(num_joints):
        plt.plot(joint_configs_array[:, i], label=f'Joint {i+1}')

    plt.title('Joint Angle Progression')
    plt.xlabel('Iteration')
    plt.ylabel('Joint Angle (radians)')
    plt.legend()
    plt.grid(True)
    plt.show()


plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['figure.dpi'] = 90
plt.ioff()

def animate_solution(path):
    fig, ax = plt.subplots()
    def animate(frame):
        ax.cla()
        for i in range(len(path[frame])):
            p.resetJointState(robot_id, i, path[frame][i], physicsClientId=sim_id)        
        showImage(ax)
    anim = animation.FuncAnimation(fig, animate, frames=len(path))
    plt.close(fig)
    return anim

In [28]:
sim_id = p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

plane_id = p.loadURDF('plane.urdf', physicsClientId=sim_id)

robot_id = p.loadURDF("kuka_iiwa/model.urdf",
            basePosition=[0,0,0],
            baseOrientation=p.getQuaternionFromEuler([0,0,0]),
            useFixedBase=True,
            globalScaling=1,
            flags=p.URDF_USE_IMPLICIT_CYLINDER,
            physicsClientId=sim_id)

num_joints = p.getNumJoints(robot_id, physicsClientId=sim_id)

In [29]:
def showImage(ax, cameraPos=[2, 2, 2]):
  viewMatrix = p.computeViewMatrix(
              cameraEyePosition=cameraPos,
              cameraTargetPosition=[0, 0, 0],
              cameraUpVector=[0, 0, 1],
              physicsClientId=sim_id)
  projectionMatrix = p.computeProjectionMatrixFOV(
              fov=60.0,
              aspect=1.0,
              nearVal=0.1,
              farVal=10,
              physicsClientId=sim_id)
  width, height, rgbImg, depthImg, segImg = p.getCameraImage(
              width=512,
              height=512,
              viewMatrix=viewMatrix,
              projectionMatrix=projectionMatrix,
              physicsClientId=sim_id)
  ax.imshow(rgbImg)

fig, ax = plt.subplots()
showImage(ax)

# Part 1: Forward Kinematics

For a pre-loaded robot model, Pybullet is already able to compute the forward kinematics of each link and the end effector. Specifically, `p.getLinkState(robot_id, link_id, computeForwardKinematics=1, sim_id)` returns a tuple, where the first element is the link position and the second element is the link orientation in quaternion form, both relative to the world frame.

## 1.1: Absolute Transformations (6 pts)

Extract and store the absolute homogeneous transformations $T^0_i$ for all frames $i \in \{1,...,7\}$. Pybullet uses 0-indexing here, so the first joint frame has index $0$ and the end effector frame has index $6$.

Recall that you can convert the quaternion into a rotation matrix using `p.getMatrixFromQuaternion()`.

In [30]:
# TODO: Find and store all absolute homogeneous transformation matrices


## 1.2: Relative Transformations (6 pts)

Given the *absolute* transformations above, we can then compute the *relative* transformation matrices $T^{i-1}_i$ as follows:

$$ T^{i-1}_i = (T^0_{i-1})^{-1} T^0_i $$

Compute each matrix $T^{i-1}_i$ and print out them out. Also separately print out the corresponding translation and rotation, the latter as a set of $xyz$ Euler angles.

To obtain the latter from a rotation matrix, you can use `scipy.spatial.transform.Rotation.from_matrix(rotation_matrix).as_euler('xyz', degrees=True)`.

In [31]:
# TODO: Compute and print all relative homogeneous transformation matrices
# Also print out the translation vectors and xyz Euler angles for rotation


# Part 2: Jacobian and Numerical IK

Now we will investigate the robot's Jacobian and use it to solve the inverse kinematics problem. We can retrieve the robot's instantaneous numerical Jacobian by calling `p.calculateJacobian()` with the usage shown in the following code cell:

In [35]:
q = [p.getJointState(robot_id, i)[0] for i in range(num_joints)]

linear_jacobian, angular_jacobian = p.calculateJacobian(
    robot_id,
    linkIndex=num_joints-1, # the link for which we want the Jacobian
    localPosition=[0,0,0],  # the point, relative to the local frame, for which we want the Jacobian
    objPositions=q,         # robot's joint configurations
    objVelocities=[0]*num_joints,    # Joint velocity list, usually 0 for kinematics
    objAccelerations=[0]*num_joints, # Joint acceleration list, usually 0
    physicsClientId=sim_id
)

This returns two Jacobian matrices, one corresponding to linear velocities and the other two angular velocities. We will only be concerned with the former in this assignment.

## 2.1: Jacobian and Singularity Analysis (6 pts)

Complete the following tasks and answer the corresponding questions.

1. Use `p.resetJointState()` to set all joint angles to $\mathbf q = \mathbf 0$ and compute the linear velocity Jacobian. Convert it into matrix form, round it to several decimal places, and print it.

2. What is the rank of the Jacobian? Which joints are able to produce nonzero motion at the end effector? Along which directions or in what space does each of those motions lie?

3. Repeat the above analysis for the configuration $\mathbf q = (0, 0, 0, \pi/2, 0, 0, 0)$.

In [37]:
# TODO: Analyze Jacobian at q = 0


In [0]:
# TODO: Analyze Jacobian at q = (0,0,0,pi/2,0,0,0)


## 2.2: Numerical Inverse Kinematics (8 pts)

Now that we have an understanding of the Jacobian, we will implement the Newton-Raphson method with damping for numerical IK. Complete the `numerical_ik()` function below. Given a `target_pos` for the end effector, it should run interatively for at most `max_iter` iterations or until the error becomes smaller than `tol`, whichever occurs first.

In each iteration, find the current end effector position, Jacobian, and position error. Compute the *damped* Jacobian pseudoinverse as $J^\top (JJ^\top + \lambda^2 I)^{-1}$, where $\lambda^2$ is provided as `lam2`. Apply the corresponding update to the configuration with step size `step`, and reset the robot to the new configuration.

Finally, your function should store a list of the updated configuration in each iteration. Return this list when the function finishes.

In [38]:
# TODO: Complete the numerical_ik() method
def numerical_ik(target_pos, lam2, step=0.1, max_iter=100, tol=1e-3):
    """
    Solves IK for target position using damped Jacobian Pseudo-Inverse.
    Updates the simulated robot's configuration at each iteration of the IK search.
    Returns a list of joint configurations computed in each step.
    """
    return []

## 2.3 Testing IK (4 pts)

The following method can be invoked to test your implementation. The robot starts at the zero configuration, and we specify a workspace target. We run IK with a given damping parameter, plot the joint angle trajectories, and verify the position reached by the end effector. Finally, we plot the robot's joints over time, and we also return the joint trajectories as a list.

In [48]:
def test_ik(lam2):
    for i in range(num_joints):
        p.resetJointState(robot_id, i, 0, physicsClientId=sim_id)

    target = np.array([0.4, 0.3, 0.2])
    solution = numerical_ik(target, lam2)
    actual_pos = p.getLinkState(robot_id, 6, computeForwardKinematics=1, physicsClientId=sim_id)[0]
    print(f"Target: {np.round(target,3)}, Reached: {np.round(actual_pos,3)}")

    plot_joint_progression(solution)
    return solution

Run `test_ik()` twice, first with $\lambda^2=0$ then with $\lambda^2 = 0.1$. Show both plots and use `animate_solution()` to visualize the robot's movement. (You can generate them in separate cells. Make sure that the animations do not cover the plots in your submission.) Briefly compare the two results and give an explanation for your observations, particularly with regard to the impact of singularities.

In [40]:
# TODO: Run test_ik() with lambda^2 = 0


In [42]:
# TODO: Run test_ik() with lambda^2 = 0.1


# Part 3: Feedback Control

For real robots, we typically do not set joint positions directly as we have been doing with `p.resetJointState()`. Instead, we command velocities or torques. To command velocities, we can call `p.setJointMotorControlArray()` as shown in the following cell:

In [45]:
dq = np.zeros(num_joints)

p.setJointMotorControlArray(
    robot_id,
    jointIndices=range(num_joints),   # which joints to command (can be some or all)
    controlMode=p.VELOCITY_CONTROL,   # alternatively, p.TORQUE_CONTROL
    targetVelocities=dq,              # list of target velocities
    physicsClientId=sim_id
)

## 3.1: P Controller (8 pts)

Write a simple P controller for velocity inputs to the robot in order to achieve a fixed target configuration. As with numerical IK, this will be implemented in discrete steps.

In each iteration, retrieve the current configuration and compute the position error. Then apply the control law to compute the desired velocity. Remember that for a multi-joint robot, we have a gain matrix, proportional to the identty matrix.

Command the velocities to the robot joints, and call `p.stepSimulation(sim_id)` to effect physical movement before ending the current iteration. Finally, as with numerical IK, your function should store a list of the updated configuration in each iteration. Return this list when the function finishes.

In [149]:
# TODO: Complete the p_control() method
def p_control(target_q, Kp, max_iter=500, tol=1e-3):
    """
    P joint position controller given fixed target.
    Returns a list of joint configurations computed in each step.
    """
    return []

Run the following code cell to test your implementation. The robot starts at the zero configuration, and we specify a target configuration. You can again verify that the animation produces expected robot motion.

In [151]:
for i in range(num_joints):
    p.resetJointState(robot_id, i, 0, physicsClientId=sim_id)

Kp = 5.0
target_q = np.array([1.0, 0.8, -0.4, -1.2, 0.0, 0.5, -0.6])
solution = p_control(target_q, Kp)

final_states = [p.getJointState(robot_id, i, sim_id) for i in range(num_joints)]
final_q_reached = np.array([s[0] for s in final_states])

print(f"Final q (target): {np.round(target_q, 3)}")
print(f"Final q (reached): {np.round(final_q_reached, 3)}")
print(f"Final position error norm: {np.linalg.norm(target_q - final_q_reached):.4f}")

plot_joint_progression(solution)
animate_solution(solution)

## 3.2: PI Controller (8 pts)

Now let's suppose we want to track a moving target configuration. A PI controller is typically preferred for this task, since the integral term can better handle steady-state error.

Implement this controller in the function below. It should share a similar structure to the P controller, with a few key differences:
- `target_q` is a list of target positions.
- The number of steps to run the controller is equal to the length of `target_q`.
- The "integral" is updated discretely in each step by an increment of `q_error * dt`.

The value of `dt` is $240$ Hz, because each call of `p.stepSimulation()` advances time by $\frac1{240}$ seconds. (It is also possible to change this in Pybullet, but we will not worry about it here.)

In [126]:
# TODO: Complete the pi_control() method
def pi_control(target_q, Kp, Ki, dt=1.0/240.0):
    """
    PI joint position controller given list of target configurations.
    Returns a list of joint configurations computed in each step.
    """
    return []

## 3.3 Testing PI Controller (4 pts)

The following method can be invoked to test your implementation. The robot starts at the zero configuration, and we specify a list of target configurations at constant velocity.

In [143]:
def test_pi(Kp, Ki):
    for i in range(num_joints):
        p.resetJointState(robot_id, i, 0, physicsClientId=sim_id)

    num_steps = 500
    q_start = np.zeros(num_joints)
    q_final = np.array([1.0, 0.8, -0.4, -1.2, 0.0, 0.5, -0.6])
    target_q = np.linspace(q_start, q_final, num_steps, endpoint=True)

    solution = pi_control(target_q, Kp, Ki)
    final_states = [p.getJointState(robot_id, i, sim_id) for i in range(num_joints)]
    final_q_reached = np.array([s[0] for s in final_states])

    print(f"Final q (target): {np.round(target_q[-1], 3)}")
    print(f"Final q (reached): {np.round(final_q_reached, 3)}")
    print(f"Final position error norm: {np.linalg.norm(target_q[-1] - final_q_reached):.4f}")

    plot_joint_progression(solution)
    return solution

Run `test_pi()` twice, first with $K_p=1$ and $K_i=0$, and then with $K_p=1$ and $K_i=5$. Note that the former is just a P controller with no integral term. Show both plots. Briefly compare the two results and give an explanation for your observations, particularly on the differences in performance for this trajectory-tracking task.

In [154]:
# TODO: Run test_pi() with Kp = 1 and Ki = 0


In [155]:
# TODO: Run test_pi() with Kp = 1 and Ki = 5
